In [ ]:
import pickle
from ruamel.yaml import YAML
from typing import Literal
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import resize

from topostats.damage.classes import GrainCollection
from topostats.plottingfuncs import Colormap

CMAP = Colormap().get_cmap()
VMIN = -3
VMAX = 4

yaml = YAML(typ="unsafe", pure=True)

In [ ]:
grain_collection_paths: list[tuple[Path, str]] = [
    (Path("/Users/sylvi/topo_data/dna_damage_cache") / "analysis_results" / "grain_collection.pkl", "cesium"),
    (Path("/Users/sylvi/topo_data/dna_damage_proton_cache") / "analysis_results" / "grain_collection.pkl", "proton"),
]

dataset_save_path = Path("/Users/sylvi/topo_data/dna-damage-unet/data")
assert dataset_save_path.exists(), f"Dataset save path {dataset_save_path} does not exist"

dataset_image_size = 512  # size of the images to be saved for training

for grain_collection_path, dataset_name in grain_collection_paths:
    assert grain_collection_path.exists(), f"Grain collection path {grain_collection_path} does not exist"
    print(f"Loading grain collection from {grain_collection_path} for dataset {dataset_name}")
    with open(grain_collection_path, "rb") as f:
        grain_collection: GrainCollection = pickle.load(f)
        # for type checking, ensure that grain_collection is of type GrainCollection
        assert isinstance(grain_collection, GrainCollection), f"Expected GrainCollection, got {type(grain_collection)}"

        for i, grain in grain_collection.grains.items():
            grain_id = grain.global_grain_id
            grain_filename = grain.filename
            sample_type = grain.sample_type
            folder = grain.folder
            image = grain.image
            original_image_shape = image.shape
            pixel_to_nm_scaling = grain.pixel_to_nm_scaling

            # resize to the new dataset image size, and calculate the new pixel to nm scaling
            image = resize(image, (dataset_image_size, dataset_image_size), anti_aliasing=True)
            # calculate the new pixel to nm scaling
            # note that higher p2nm means lower resolution
            # 64x64 @ 0.5 nm/pixel = 32x32 nm
            # 128x128 @ 0.25 nm/pixel = 32x32 nm
            # so if the image is upscaled, the pixel to nm scaling should be downscaled, and vice versa
            # since there are more pixels per nm
            old_to_new_image_size_ratio = original_image_shape[0] / dataset_image_size
            new_pixel_to_nm_scaling = pixel_to_nm_scaling * old_to_new_image_size_ratio

            plt.imshow(image, cmap=CMAP, vmin=VMIN, vmax=VMAX)

            filename = f"{grain_filename}"

            # Save the image to a png file for labelling in label studio
            plt.imsave(f"{dataset_save_path}/{filename}.png", image, cmap=CMAP, vmin=VMIN, vmax=VMAX)
            plt.close()

            # Save the image to a npy file for training
            np.save(f"{dataset_save_path}/{filename}_image.npy", image)

            # Save the metadata as a yaml file
            metadata = {
                "grain_id": grain_id,
                "original_pixel_to_nm_scaling": pixel_to_nm_scaling,
                "new_pixel_to_nm_scaling": new_pixel_to_nm_scaling,
                "image_shape": list(image.shape),
                "folder": folder,
                "sample_type": sample_type,
                "bbox": list(grain.bbox),
                "filename": grain.filename,
                "percent_damage": grain.percent_damage,
                "dataset_name": dataset_name,
                "original_image_shape": list(original_image_shape),
                "image_shape": list(image.shape),
            }
            with open(f"{dataset_save_path}/{filename}_metadata.yaml", "w") as f:
                yaml.dump(metadata, f)